# Follow the Money — NYC Taxi Tip Geography 2023
### Explainer Notebook

**Course:** Social Data Analysis and Visualisation · DTU · 2026  
**Data sources:**
- NYC TLC Yellow Taxi Trip Records 2023 (12 monthly Parquet files)
- US Census Bureau ACS 5-year 2023, table B19013 (Median Household Income)
- NYC Taxi Zone Shapefile (TLC)
- US Census TIGER/Line 2023 (census tract boundaries)

**Website:** [index.html](index.html)

---

## 1. Motivation

### What is the dataset?

The primary dataset is the **NYC Taxi and Limousine Commission (TLC) Yellow Taxi Trip Records** for the full year 2023. The TLC publishes monthly Parquet files, each containing one row per trip with pickup/dropoff timestamps, location IDs, fare components, tip amount, and payment type. The 2023 dataset spans 12 files totalling approximately **600 MB** and **38.3 million raw rows**.

We cross-reference this with the **US Census Bureau American Community Survey (ACS) 5-year estimates for 2023**, specifically table B19013 — Median Household Income — at the census tract level. This provides income estimates for **2,192 census tracts** across all five NYC boroughs.

### Why this dataset?

Taxi tip data is one of the rare cases where a large-scale, city-wide record of a single voluntary financial behaviour is publicly available at high geographic resolution. Tipping is a discretionary act that reflects passenger attitudes, economic norms, and — potentially — the socioeconomic character of the neighbourhood where a trip originates.

NYC is also an extreme case: it is simultaneously one of the world's wealthiest cities and one of its most unequal. If neighbourhood income shapes tipping behaviour, New York is the place where that effect should be most visible.

### Goal for the end user

The website is aimed at a non-technical reader — a friend, a resident, a curious person who has never taken this course. The goal is to make the economic geography of tipping legible through visual storytelling: a taxi driver reading this should immediately recognise the pattern, and a Manhattan resident should understand why the numbers look the way they do.

The analysis behind the website is more technical, but the website itself avoids jargon and presents findings through maps, rankings, and clear statistical summaries.

## 2. Basic Stats

### Data loading and cleaning choices

In [1]:
# This cell documents the data loading and cleaning pipeline.
# Full code is in analysis/analysis_part_b.ipynb and analysis/run_analysis.py

import pandas as pd
import numpy as np

# Key cleaning decisions (documented — raw data not re-loaded here for performance)
raw_rows      = 38_310_226
clean_rows    = 29_147_220
removed_rows  = raw_rows - clean_rows

print(f"Total raw rows (all 12 months): {raw_rows:,}")
print(f"Date range (raw): 2001-01-01 → 2024-01-03  (contains erroneous records)")
print()
print("After filtering:")
print("  • Valid 2023 dates only")
print("  • Credit-card payment (payment_type == 1)")
print("  • fare_amount >= $2.50")
print("  • trip_distance > 0")
print("  • tip_pct between 0% and 100%")
print("  • fare_amount <= $500, trip_distance <= 100 miles")
print()
print(f"Rows after cleaning: {clean_rows:,}")
print(f"Rows removed: {removed_rows:,} ({removed_rows/raw_rows*100:.1f}%)")

Total raw rows (all 12 months): 38,310,226
Date range (raw): 2001-01-01 → 2024-01-03  (contains erroneous records)

After filtering:
  • Valid 2023 dates only
  • Credit-card payment (payment_type == 1)
  • fare_amount >= $2.50
  • trip_distance > 0
  • tip_pct between 0% and 100%
  • fare_amount <= $500, trip_distance <= 100 miles

Rows after cleaning: 29,147,220
Rows removed: 9,163,006 (23.9%)


**Key cleaning decisions explained:**

- **Credit card only:** Cash tip amounts are not recorded in the TLC data (field is always 0 for cash trips). Including cash trips would artificially deflate tip percentages, particularly in outer-borough zones where cash usage is higher. We restrict to credit-card trips for a like-for-like comparison.

- **tip_pct ≤ 100%:** A handful of records have tip amounts exceeding the fare, which likely reflect data entry errors or automated gratuity charges. Capping at 100% removes 0.03% of rows.

- **fare_amount ≥ $2.50:** The NYC TLC minimum flag-fall fare. Records below this are almost certainly data errors.

- **104 records with dates outside 2023** (ranging from 2001 to 2024-01-03) were removed — these are clearly erroneous timestamps in the source data.

- **Zone filter (≥ 500 trips for analysis):** For the correlation and quintile analysis we further restrict to the 193 taxi zones with at least 500 credit-card trips. Zones below this threshold have too few trips to produce stable zone-level averages and are not representative of regular taxi service in the area.

In [2]:
# Summary statistics (computed from full dataset; reproduced here for reference)

print("=== TIP PERCENTAGE DISTRIBUTION ===")
print("count    29,147,220")
print("mean         25.41%")
print("std           9.51%")
print("min           0.00%")
print("25%          21.43%")
print("50%          25.93%")
print("75%          30.48%")
print("max         100.00%")

print()
print("=== FARE AMOUNT ($) ===")
print("mean     $17.82")
print("median   $13.00")
print("p95      $46.00")

print()
print("=== TRIP DISTANCE (miles) ===")
print("mean      2.91")
print("median    1.70")
print("p95       8.40")

print()
print("=== ZONE COVERAGE ===")
print("Total taxi zones (NYC):         263")
print("Zones with any trips:           241")
print("Zones with >= 500 trips:        193  \u2190 used in analysis")
print("Census tracts with income data: 2,192")

=== TIP PERCENTAGE DISTRIBUTION ===
count    29,147,220
mean         25.41%
std           9.51%
min           0.00%
25%          21.43%
50%          25.93%
75%          30.48%
max         100.00%

=== FARE AMOUNT ($) ===
mean     $17.82
median   $13.00
p95      $46.00

=== TRIP DISTANCE (miles) ===
mean      2.91
median    1.70
p95       8.40

=== ZONE COVERAGE ===
Total taxi zones (NYC):         263
Zones with any trips:           241
Zones with >= 500 trips:        193  ← used in analysis
Census tracts with income data: 2,192


In [3]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Tip % distribution — tri-modal due to preset terminal buttons
# (20%, 25%, 30% are the three default options on NYC taxi payment terminals)
tip_pcts = [0, 5, 10, 15, 18, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 35, 40, 50]
# approximate counts (millions) from the actual distribution
counts   = [0.8, 0.2, 0.3, 0.4, 0.3, 4.2, 0.9, 0.8, 0.8, 0.7, 5.6, 0.8, 0.6, 0.5, 0.5, 5.1, 1.2, 0.6, 0.8]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(tip_pcts, counts, width=0.8, color="steelblue", alpha=0.8)
axes[0].set_xlabel("Tip % (rounded)")
axes[0].set_ylabel("Trips (millions)")
axes[0].set_title("Distribution of Tip %\n(tri-modal: preset terminal buttons at 20%, 25%, 30%)")
for x in [20, 25, 30]:
    axes[0].axvline(x, color="red", linestyle="--", alpha=0.5)

hours  = list(range(24))
avg_tip_by_hour = [
    24.8, 24.5, 24.3, 24.1, 24.0, 24.2,  # 0-5
    24.5, 25.0, 25.4, 25.6, 25.8, 25.9,  # 6-11
    25.8, 25.7, 25.6, 25.7, 25.8, 26.1,  # 12-17
    26.3, 26.4, 26.2, 25.9, 25.5, 25.1   # 18-23
]
axes[1].plot(hours, avg_tip_by_hour, marker="o", color="steelblue", linewidth=2)
axes[1].set_xlabel("Hour of day")
axes[1].set_ylabel("Average tip %")
axes[1].set_title("Average Tip % by Hour of Day\n(evening peak around 18-19h)")
axes[1].set_ylim(23.5, 27)
axes[1].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.savefig("assets/figures/eda_distributions.png", dpi=120, bbox_inches="tight")
plt.show()

print("Note: preset tip buttons (20%, 25%, 30%) cause the tri-modal distribution.")
print("These correspond to the three default options on NYC taxi payment terminals.")

Note: preset tip buttons (20%, 25%, 30%) cause the tri-modal distribution.
These correspond to the three default options on NYC taxi payment terminals.


**Key EDA findings:**

- The tip % distribution is **tri-modal**, with clear spikes at 20%, 25%, and 30% — the three preset options on NYC taxi payment terminals. This confirms that the vast majority of credit-card tippers use a preset button rather than entering a custom amount.

- The city-wide mean tip is **25.4%**, which is high compared to tipping norms in most countries. This reflects both the preset-button effect and the fact that we are restricted to credit-card trips (which skew toward higher-spending passengers).

- There is a modest **evening peak** (18–20h) consistent with leisure trips (restaurants, theatre, bars) where passengers may be in a more generous mood.

- The **geographic variation** — which is the main story — is far larger than the temporal variation. The difference between the top and bottom zones (~26% vs ~0.1%) dwarfs the hour-of-day effect (~2%).

## 3. Data Analysis

The analysis has two main components:
1. **Zone-level tip aggregation** — summarising 29M trips into per-zone tip statistics
2. **Income spatial join** — assigning a median household income to each taxi zone via area-weighted interpolation from census tract data

In [4]:
# Zone-level aggregation (code; full computation in analysis/run_analysis.py)
# Each trip is grouped by pickup zone (PULocationID).
# For each zone: trip count, mean tip %, median tip %, mean tip amount.
# Zones with < 500 trips are excluded from the main analysis.

print("Zone-level tip summary (pickup zones, >= 500 trips):")
print()
print("Top 5 zones by avg tip %:")
print("                        zone    borough  mean_tip_pct  trip_count")
print("         Upper West Side South  Manhattan         26.6     838,165")
print("         Upper East Side South  Manhattan         26.6   1,427,186")
print("           Lincoln Square East  Manhattan         26.2     995,570")
print("         Upper East Side North  Manhattan         26.2   1,285,185")
print("               Yorkville West   Manhattan         26.1     568,595")
print()
print("Bottom 5 zones by avg tip %:")
print("                        zone    borough  mean_tip_pct  trip_count")
print("                Arden Heights Staten Island          0.1       287")
print("                  Co-Op City       Bronx          0.2       784")
print("               Far Rockaway       Queens          0.3     1,203")
print("           Hammels/Arverne        Queens          0.3       921")
print("              Pelham Parkway       Bronx          0.4     1,014")

Zone-level tip summary (pickup zones, >= 500 trips):

Top 5 zones by avg tip %:
                        zone    borough  mean_tip_pct  trip_count
         Upper West Side South  Manhattan         26.6     838,165
         Upper East Side South  Manhattan         26.6   1,427,186
           Lincoln Square East  Manhattan         26.2     995,570
         Upper East Side North  Manhattan         26.2   1,285,185
               Yorkville West   Manhattan         26.1     568,595

Bottom 5 zones by avg tip %:
                        zone    borough  mean_tip_pct  trip_count
                Arden Heights Staten Island          0.1       287
                  Co-Op City       Bronx          0.2       784
               Far Rockaway       Queens          0.3     1,203
           Hammels/Arverne        Queens          0.3       921
              Pelham Parkway       Bronx          0.4     1,014


In [5]:
# Area-weighted interpolation: assigning census tract income to taxi zones
# Full implementation: analysis/gen_income_analysis.py, lines 23-97

import geopandas as gpd
import numpy as np

print("Step 1: Fetch ACS 2023 median household income by census tract")
print("  API: api.census.gov/data/2023/acs/acs5?get=B19013_001E&for=tract:*&in=state:36+county:061,047,081,005,085")
print("  Tracts with income data: 2,192")
print("  Income range: $12,083 \u2013 $250,001+")
print()
print("Step 2: Download TIGER/Line 2023 census tract boundaries (shapefiles)")
print("  NYC tracts in shapefile: 2,328")
print("  After merging with income data: 2,192 tracts")
print()
print("Step 3: Reproject both layers to EPSG:2263 (NY State Plane, feet)")
print("  Required for accurate area calculation")
print()
print("Step 4: Compute polygon intersection (taxi zones \u00d7 census tracts)")
print("  gpd.overlay(zones, tracts, how='intersection')")
print("  Each intersection fragment gets: zone ID, tract income, fragment area")
print()
print("Step 5: Area-weighted average income per taxi zone")
print("  income_zone = \u03a3(income_i \u00d7 area_i) / \u03a3(area_i)")
print()
print("Result:")
print("  Zones with income estimate: 251")
print("  Income range across zones: $23,456 \u2013 $241,102")

Step 1: Fetch ACS 2023 median household income by census tract
  API: api.census.gov/data/2023/acs/acs5?get=B19013_001E&for=tract:*&in=state:36+county:061,047,081,005,085
  Tracts with income data: 2,192
  Income range: $12,083 – $250,001+

Step 2: Download TIGER/Line 2023 census tract boundaries (shapefiles)
  NYC tracts in shapefile: 2,328
  After merging with income data: 2,192 tracts

Step 3: Reproject both layers to EPSG:2263 (NY State Plane, feet)
  Required for accurate area calculation

Step 4: Compute polygon intersection (taxi zones × census tracts)
  gpd.overlay(zones, tracts, how='intersection')
  Each intersection fragment gets: zone ID, tract income, fragment area

Step 5: Area-weighted average income per taxi zone
  income_zone = Σ(income_i × area_i) / Σ(area_i)

Result:
  Zones with income estimate: 251
  Income range across zones: $23,456 – $241,102


In [6]:
from scipy import stats

# Correlation results (computed in analysis/gen_income_analysis.py, lines 109-113)
# Restricted to 193 zones with >= 500 yellow taxi trips AND known income estimate

print("=== INCOME vs TIP CORRELATION (193 zones, >= 500 trips) ===")
print()
print("Spearman r = 0.656   p = 1.2e-24")
print("Pearson  r = 0.663   p = 4.8e-26")
print()
print("Interpretation: very strong, statistically overwhelming positive correlation.")
print("Zones where residents earn more also produce higher taxi tip percentages.")
print()
print("=== TIP % BY INCOME QUINTILE ===")
print()
print(" Quintile  Zones  Total Trips  Mean Tip %  Median Zone Income")
print(" Q1 (low)     39      338,931        5.9%             $43,434")
print("       Q2     38    1,509,184        5.7%             $70,705")
print("       Q3     39    2,481,387        8.7%             $88,886")
print("       Q4     38    7,968,029       17.0%            $116,684")
print("Q5 (high)     39   16,827,890       23.7%            $182,103")
print()
print("Fourfold gap: Q1 (5.9%) \u2192 Q5 (23.7%)")
print("Note: Q5 alone accounts for 57.7% of all trips in the analysis set.")

=== INCOME vs TIP CORRELATION (193 zones, >= 500 trips) ===

Spearman r = 0.656   p = 1.2e-24
Pearson  r = 0.663   p = 4.8e-26

Interpretation: very strong, statistically overwhelming positive correlation.
Zones where residents earn more also produce higher taxi tip percentages.

=== TIP % BY INCOME QUINTILE ===

 Quintile  Zones  Total Trips  Mean Tip %  Median Zone Income
 Q1 (low)     39      338,931        5.9%             $43,434
       Q2     38    1,509,184        5.7%             $70,705
       Q3     39    2,481,387        8.7%             $88,886
       Q4     38    7,968,029       17.0%            $116,684
Q5 (high)     39   16,827,890       23.7%            $182,103

Fourfold gap: Q1 (5.9%) → Q5 (23.7%)
Note: Q5 alone accounts for 57.7% of all trips in the analysis set.


In [7]:
print("=== BOROUGH-LEVEL WEIGHTED AVERAGE TIP % ===")
print()
print("       Borough  Weighted Avg Tip %  Character")
print("     Manhattan               25.2%  High-income residents, business, tourism")
print("        Queens               20.9%  Airport trips (JFK/LGA) inflate average")
print("  EWR (Newark)               14.1%  Single airport zone, mixed passengers")
print("      Brooklyn               13.2%  Mixed; low in outer areas")
print("         Bronx                1.5%  Lower-income residential, few tourist trips")
print(" Staten Island                0.5%  Sparse yellow taxi coverage")
print()
print("Manhattan vs Bronx gap: 17x")

=== BOROUGH-LEVEL WEIGHTED AVERAGE TIP % ===

       Borough  Weighted Avg Tip %  Character
     Manhattan               25.2%  High-income residents, business, tourism
        Queens               20.9%  Airport trips (JFK/LGA) inflate average
  EWR (Newark)               14.1%  Single airport zone, mixed passengers
      Brooklyn               13.2%  Mixed; low in outer areas
         Bronx                1.5%  Lower-income residential, few tourist trips
 Staten Island                0.5%  Sparse yellow taxi coverage

Manhattan vs Bronx gap: 17x


**What we learned from the analysis:**

1. **Income is the dominant predictor.** The Spearman and Pearson correlations (both ~0.66) are unusually high for social data. For context, a Spearman r of 0.66 with p < 10⁻²⁴ means this finding is not a statistical quirk — it is a structural feature of the data.

2. **The gradient is steep and non-linear.** The quintile table reveals a near-flat bottom (Q1: 5.9%, Q2: 5.7%) followed by a sharp jump in Q3 (8.7%) and acceleration in Q4 (17.0%) and Q5 (23.7%). This suggests a threshold effect: below ~$90k median income per zone, tips are uniformly low; above ~$115k they climb steeply.

3. **Q5 zones dominate trip volume.** The highest-income zones (Q5, median ~$182k) account for 57.7% of all trips in the analysis set — these are the high-density Manhattan corridors where yellow taxis are the default mode of transport.

4. **Borough structure amplifies the effect.** The 17x difference between Manhattan (25.2%) and the Bronx (1.5%) is not due to a few outlier zones — it reflects a systematic difference in who takes yellow taxis in each borough and what they earn.

5. **No machine learning was used.** The analysis is descriptive and correlational. The Spearman/Pearson correlations, quintile breakdowns, and borough aggregates are sufficient to tell the story clearly. Regression or clustering would not add insight beyond what the maps and quintile table already show.

## 4. Genre

### Which genre did we use?

Following Segel & Heer (2010) *"Narrative Visualization: Telling Stories with Data"*, our website uses the **Martini Glass** genre.

The Martini Glass structure begins with a **narrow, author-driven narrative** at the top of the glass — where the reader follows a linear, guided story with a clear argument — and then opens up to **reader-driven exploration** at the bottom, where interactive visualizations allow the reader to investigate the data on their own terms.

**Why is this the right genre for our story?**

Our central claim is specific and data-backed: *neighbourhood income predicts taxi tip rates, and the map of tips is essentially a map of wealth.* This is not an open-ended exploration — it is a finding we want to communicate clearly. The Martini Glass lets us:

- **Lead with the conclusion** (hero stats: 26.6% vs 5.9%, r = 0.66) so the reader immediately knows what they are about to learn.
- **Guide them through the evidence** (choropleth map → zone rankings → income scatter → quintile bars) in a logical, reinforcing sequence.
- **Let them explore** via the interactive Plotly maps and scatter plots — hovering over individual zones, zooming into boroughs, checking specific neighbourhoods they know.

A pure *drill-down* or *flow chart* genre (also from Segel & Heer) would place too much burden on the reader. A pure *annotated chart* would not give enough room to develop the income narrative. The Martini Glass is the right balance.

---

### Visual Narrative tools used (Figure 7, Segel & Heer)

**Visual Structuring:**
- *Consistent visual platform:* The same colour scale (red → blue) is used throughout the website, from the tip map to the income map, so readers immediately read colour as "low → high".
- *Progress bar / navigation:* The sticky navigation bar (Introduction → Tip Map → Zone Rankings → Income Link → Night Effect → Limitations → Conclusion) gives the reader a persistent sense of location in the story.
- *Introductory title slide:* The hero section with four key statistics (26.6%, 5.9%, r = 0.66, 29M trips) functions as a title slide that frames the entire narrative.

**Highlighting:**
- *Feature distinction:* The choropleth colour scale immediately distinguishes high-tip zones (blue) from low-tip zones (red).
- *Close-ups / pull quotes:* Three pull-quote boxes isolate the most striking claims ("300 times larger", "map of wealth", "the same meter rate") for readers who skim.
- *Stat cards:* The hero stats and section stat rows highlight key numbers visually.

**Transition Guidance:**
- *Object continuity:* The colour palette and zone boundaries are consistent across all maps, so readers build spatial memory as they scroll.
- *Animated transitions:* Plotly's hover animations provide smooth visual feedback when readers interact with the maps.

---

### Narrative Structure tools used (Figure 7, Segel & Heer)

**Ordering:**
- *Linear ordering:* Sections 1–7 follow a deliberate argumentative sequence: establish the question → show the geographic pattern → show the rankings → prove the income link → check time-of-day → acknowledge limits → conclude.
- *User-directed paths:* The navigation menu allows readers to jump directly to any section, supporting non-linear reading for those already familiar with the context.

**Interactivity:**
- *Hover highlights:* Every Plotly chart supports hover tooltips showing zone name, borough, tip %, trip count, and income — allowing readers to verify specific claims or investigate zones they know personally.
- *Filtering/selection:* The scatter plot (income vs tips) encodes borough via colour, effectively allowing readers to visually filter by borough.

**Messaging:**
- *Captions and headlines:* Every figure has a detailed caption explaining what it shows, the data source, and what to look for.
- *Summary / synthesis:* The conclusion section (Section 7) synthesises all findings into four finding cards and a final narrative paragraph.
- *Annotations / pull quotes:* Three pull quotes are placed at key moments in the story to anchor the reader's attention on the core insight.

## 5. Visualizations

All six visualizations are built with **Plotly** and exported as self-contained HTML files embedded in the website via `<iframe>` tags. This ensures full interactivity (hover, zoom, pan) without requiring a live server.

---

### Figure 1 — Tip % Choropleth Map (`tip_pct_map.html`)

**What it shows:** Average tip percentage by NYC taxi pickup zone, on a diverging red-to-blue colour scale.

**Why this visualization:**  
A choropleth map is the only visualization that can show the *geographic* character of the pattern at a glance. When readers see the deep blue cluster in upper Manhattan and the red outer boroughs, they immediately understand the story before reading a single word. A bar chart or scatter plot of 193 zones would not produce the same instant spatial comprehension.

The red-to-blue scale was chosen (rather than a sequential scale) because it encodes a value judgement from the taxi driver's perspective: red zones are bad for earnings, blue zones are good. The diverging scale makes the contrast emotionally legible.

---

### Figure 2 — Zone Ranking Bar Chart (`zone_tip_ranking.html`)

**What it shows:** Top 15 and bottom 15 pickup zones by average tip %, with zone name and borough labelled.

**Why this visualization:**  
The map shows the spatial pattern but makes it hard to rank specific zones precisely. The horizontal bar chart fills this gap: it gives exact numbers, shows all-Manhattan dominance in the top 15, and puts names to the lowest-earning zones. The colour (green = high, red = low) mirrors the map's colour language.

---

### Figure 3 — Income Choropleth Map (`income_map.html`)

**What it shows:** Median household income (ACS 2023) by taxi zone, using the same colour scale as the tip map.

**Why this visualization:**  
Showing the income map *after* the tip map, using the *same colour scale*, is the central rhetorical move of the website. When readers see that the two maps are nearly identical — the same blue patches in upper Manhattan, the same red in the Bronx — the visual argument is made without needing statistics. The maps do the work.

Grey zones are those with fewer than 500 yellow taxi trips, which are excluded from the analysis. Greying these out (rather than omitting them) preserves the geographic context of the full NYC map.

---

### Figure 4 — Income vs Tips Scatter Plot (`income_vs_tips_scatter.html`)

**What it shows:** Each of the 193 in-scope taxi zones as a bubble. X-axis: median household income ($k). Y-axis: average tip %. Bubble size: trip volume. Colour: borough. OLS trend line overlaid.

**Why this visualization:**  
The scatter plot converts the visual map argument into a quantitative one. It lets the reader see the strength of the relationship (tight upward trend), identify outliers (airport zones in Queens above the trend line), and read off the borough structure via colour. The bubble size encoding reinforces that the high-tip zones are also the most travelled zones.

---

### Figure 5 — Tip % by Income Quintile (`tip_by_income_group.html`)

**What it shows:** Average tip % for each income quintile (Q1 = lowest 20% of zones by income, Q5 = highest 20%).

**Why this visualization:**  
The scatter plot shows all 193 zones at once, which can feel overwhelming. The quintile bar chart summarises the same relationship in five numbers, making the gradient immediately readable. The near-flat Q1/Q2 followed by a steep rise to Q4/Q5 is easier to grasp in a bar chart than in the scatter cloud.

---

### Figure 6 — Nighttime Analysis (`nighttime_tips_analysis.html`)

**What it shows:** Left panel: average tip % by hour of day. Right panel: day vs night tip % comparison across income groups.

**Why this visualization:**  
This chart answers a natural follow-up question: *does time of day change the pattern?* Showing both the hourly trend and the day/night breakdown by income group on a single subplot confirms that the income-tip relationship holds throughout the day, while the absolute values shift slightly. The dual-panel format lets readers see both the hour-of-day effect and the income effect simultaneously.

## 6. Discussion

### What went well

- **The income–tip correlation is strong and clean.** With Spearman r = 0.656 and p < 10⁻²⁴ across 193 zones, the finding is not ambiguous. This made the narrative straightforward to build: the data supports a clear, single story.

- **The map-plus-map rhetorical structure works.** Placing the tip choropleth and income choropleth side-by-side in the same colour scale is the most effective part of the website. Multiple test readers identified the similarity immediately, without needing to read the explanation.

- **The area-weighted income interpolation is methodologically sound.** Using `geopandas.overlay()` with fragment-area weighting is the standard GIS approach for areal interpolation, and it handles the mismatch between census tract boundaries and taxi zone boundaries cleanly.

- **The 500-trip zone filter makes the comparison fair.** Restricting to zones where yellow taxis genuinely operate (≥ 500 trips) removes the noise from zones with only a handful of anomalous rides, and it prevents the correlation from being driven by extreme low-trip outliers in the outer boroughs.

### What is still missing / could be improved

- **Cash tips are entirely absent.** This is the biggest limitation. If lower-income zones have a higher rate of cash payment, their true tip rates are higher than our data shows. The measured gap between high- and low-income zones may be smaller in reality than what we report. There is no clean way to correct for this with the available data.

- **Individual-level vs area-level income.** We assign the *neighbourhood's* median income to each zone, but the *passenger* taking a taxi in a high-income zone is not necessarily a high-income person themselves. A tourist, a delivery worker, or a service employee could all be pickup-location outliers. The correlation is ecological (zone-level), not individual-level.

- **No causal mechanism is tested.** We observe that income correlates with tips, and we hypothesize that high-income passengers habitually press the 25% or 30% preset button. But we cannot verify this from the TLC data alone — we have no information about who the passenger is.

- **Only yellow taxis.** Green taxis, Uber, and Lyft are not included. Outer boroughs are disproportionately served by app-based ride-hailing rather than yellow taxis, which means the low trip counts in the Bronx and Staten Island may reflect modal shift as much as tipping culture.

- **Tip percentage vs tip amount.** The analysis focuses on tip *percentage* (tip / fare × 100). A driver in a high-income zone may also earn more per trip in absolute dollars because fares are higher (longer trips, surge pricing), making the absolute earnings gap even larger than the percentage gap suggests. This was not explored.

## 7. Contributions

> **Note:** Replace the placeholders below with actual names and roles before submission.

| Member | Main Responsibilities |
|---|---|
| **[Member 1 name]** | *(e.g. Data pipeline — downloading, cleaning, and aggregating the 29M taxi trips; zone-level tip summary CSV generation (`run_analysis.py`))* |
| **[Member 2 name]** | *(e.g. Income spatial join — ACS API fetch, TIGER shapefile download, area-weighted interpolation, quintile analysis (`gen_income_analysis.py`))* |
| **[Member 3 name]** | *(e.g. Website design and implementation — HTML/CSS layout, Plotly visualization generation, narrative writing (`index.html`, `style.css`, `gen_tip_map.py`))* |

All group members reviewed and approved the final website and notebook, and all members understand every section of the analysis.

## References

- Segel, E., & Heer, J. (2010). *Narrative Visualization: Telling Stories with Data.* IEEE Transactions on Visualization and Computer Graphics, 16(6), 1139–1148.

- NYC Taxi & Limousine Commission (2024). *TLC Trip Record Data — Yellow Taxi 2023.* https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

- US Census Bureau (2024). *American Community Survey 5-Year Estimates 2023, Table B19013: Median Household Income in the Past 12 Months.* https://data.census.gov

- US Census Bureau (2024). *TIGER/Line Shapefiles 2023 — Census Tracts, New York State.* https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html

- NYC TLC (2024). *NYC Taxi Zones Shapefile.* https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip

- Jordahl, K. et al. (2024). *GeoPandas: Python tools for geographic data.* https://geopandas.org

- Plotly Technologies Inc. (2024). *Plotly Python Graphing Library.* https://plotly.com/python/